In [ ]:
import pandas as pd

courses = pd.read_csv("https://waf.cs.illinois.edu/discovery/course-catalog.csv")
print (courses.columns)
print (courses.head())

Index(['Year', 'Term', 'YearTerm', 'Subject', 'Number', 'Name', 'Description',
       'Credit Hours', 'Section Info', 'Degree Attributes',
       'Schedule Information', 'CRN', 'Section', 'Status Code', 'Part of Term',
       'Section Title', 'Section Credit Hours', 'Section Status',
       'Enrollment Status', 'Type', 'Type Code', 'Start Time', 'End Time',
       'Days of Week', 'Room', 'Building', 'Instructors'],
      dtype='str')
   Year    Term YearTerm Subject  Number                          Name  \
0  2026  Spring  2026-sp     AAS     100  Intro Asian American Studies   
1  2026  Spring  2026-sp     AAS     100  Intro Asian American Studies   
2  2026  Spring  2026-sp     AAS     100  Intro Asian American Studies   
3  2026  Spring  2026-sp     AAS     100  Intro Asian American Studies   
4  2026  Spring  2026-sp     AAS     100  Intro Asian American Studies   

                                         Description Credit Hours  \
0  Interdisciplinary introduction to the basic c

In [ ]:
courses["Credit Hours Clean"] = courses["Credit Hours"].str.extract(r"(\d+)").astype(float)

  Credit Hours  Credit Hours Clean
0     3 hours.                 3.0
1     3 hours.                 3.0
2     3 hours.                 3.0
3     3 hours.                 3.0
4     3 hours.                 3.0


In [7]:
courses["Start Time Clean"] = pd.to_datetime(
    courses["Start Time"], format="%I:%M %p", errors="coerce"
).dt.time

courses["End Time Clean"] = pd.to_datetime(
    courses["End Time"], format="%I:%M %p", errors="coerce"
).dt.time
print (courses[["Credit Hours", "Credit Hours Clean", "Start Time", "Start Time Clean", "End Time", "End Time Clean"]].head())

  Credit Hours  Credit Hours Clean Start Time Start Time Clean  End Time  \
0     3 hours.                 3.0   09:00 AM         09:00:00  09:50 AM   
1     3 hours.                 3.0   10:00 AM         10:00:00  10:50 AM   
2     3 hours.                 3.0   11:00 AM         11:00:00  11:50 AM   
3     3 hours.                 3.0   12:00 PM         12:00:00  12:50 PM   
4     3 hours.                 3.0   01:00 PM         13:00:00  01:50 PM   

  End Time Clean  
0       09:50:00  
1       10:50:00  
2       11:50:00  
3       12:50:00  
4       13:50:00  


In [12]:
def filter_courses(
    df,
    gen_ed=None,
    credits=None,
    days=None,
    part_of_term=None,
    start_time=None,
    end_time=None
):
    result = df.copy()

    # Filter by gen-ed category
    if gen_ed:
        result = result[
            result["Degree Attributes"].fillna("").str.contains(gen_ed, case=False, na=False)
        ]

    # Filter by credit hours
    if credits is not None:
        result = result[result["Credit Hours Clean"] == credits]

    # Filter by part of term
    if part_of_term:
        result = result[
            result["Part of Term"].fillna("").str.contains(part_of_term, case=False, na=False)
        ]

    # Filter by days of week
    if days:
        wanted_days = set(days)
        result = result[
            result["Days of Week"].fillna("").apply(lambda x: set(x) == wanted_days)
        ]

    # Filter by time range
    if start_time is not None:
        result = result[result["Start Time Clean"] >= start_time]

    if end_time is not None:
        result = result[result["End Time Clean"] <= end_time]

    return result

In [16]:
from datetime import time

filtered = filter_courses(
    courses,
    gen_ed="Humanities",
    credits=3,
    days=["M", "W"],
    part_of_term="1",
    start_time=time(10, 0),
    end_time=time(16, 0)
)

print(filtered[[
    "Subject", "Number", "Name",
    "Credit Hours Clean", "Days of Week",
    "Start Time", "End Time", "Degree Attributes"
]].head(20))

     Subject  Number                                               Name  \
608     AFRO     132                             African American Music   
613     AFRO     224                      Humanist Persp of Afro-Am Exp   
618     AFRO     261                      Intro to the African Diaspora   
685      AIS     101                   Intro to American Indian Studies   
691      AIS     285                                Indigenous Thinkers   
720     ANSC     120         The Art and Ethics of Animal Documentaries   
877     ANTH     261                      Intro to the African Diaspora   
1197    ARTH     110  Introduction to the History of Art and Visual ...   
1210    ARTH     220                      African Arts and Architecture   
3804      CW     100                          Intro to Creative Writing   
3849     CWL     207                           Indian Cinema in Context   
3858     CWL     226                      Humanist Persp of Afro-Am Exp   
3868     CWL     230     

In [14]:
filter_courses(courses, credits=4)


,Year,Term,YearTerm,Subject,Number,Name,Description,Credit Hours,Section Info,Degree Attributes,...,Type Code,Start Time,End Time,Days of Week,Room,Building,Instructors,Credit Hours Clean,Start Time Clean,End Time Clean
23,2026,Spring,2026-sp,AAS,310,Race and Cultural Diversity,"Same as AFRO 310, EPOL 310, and LLS 310. See E...",4 hours.,"Same as AFRO 310, EPOL 310, and LLS 310. See E...","Advanced Composition, and Cultural Studies - U...",...,DIS,02:00 PM,03:50 PM,R,17,Education Building,"Moton, T;Sultan, I",4.0,14:00:00,15:50:00
24,2026,Spring,2026-sp,AAS,310,Race and Cultural Diversity,"Same as AFRO 310, EPOL 310, and LLS 310. See E...",4 hours.,"Same as AFRO 310, EPOL 310, and LLS 310. See E...","Advanced Composition, and Cultural Studies - U...",...,DIS,02:00 PM,03:50 PM,R,42A,Education Building,"Ewa, O;Moton, T",4.0,14:00:00,15:50:00
25,2026,Spring,2026-sp,AAS,310,Race and Cultural Diversity,"Same as AFRO 310, EPOL 310, and LLS 310. See E...",4 hours.,"Same as AFRO 310, EPOL 310, and LLS 310. See E...","Advanced Composition, and Cultural Studies - U...",...,DIS,02:00 PM,03:50 PM,R,389,Education Building,"Ding, O;Moton, T",4.0,14:00:00,15:50:00
26,2026,Spring,2026-sp,AAS,310,Race and Cultural Diversity,"Same as AFRO 310, EPOL 310, and LLS 310. See E...",4 hours.,"Same as AFRO 310, EPOL 310, and LLS 310. See E...","Advanced Composition, and Cultural Studies - U...",...,DIS,02:00 PM,03:50 PM,R,241,Armory,"Moton, T;Ruiz-Divas, V",4.0,14:00:00,15:50:00
27,2026,Spring,2026-sp,AAS,310,Race and Cultural Diversity,"Same as AFRO 310, EPOL 310, and LLS 310. See E...",4 hours.,"Same as AFRO 310, EPOL 310, and LLS 310. See E...","Advanced Composition, and Cultural Studies - U...",...,DIS,02:00 PM,03:50 PM,R,242,Armory,"Moton, T;Pender, L",4.0,14:00:00,15:50:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12811,2026,Spring,2026-sp,WLOF,403,Intermediate Wolof I,"Survey of more advanced grammar, with emphasis...",4 hours.,Same as AFST 443. 4 undergraduate hours. 4 gra...,NaN,...,LCD,10:00 AM,10:50 AM,MTWRF,1140,"Literatures, Cultures, & Ling","Toure, V",4.0,10:00:00,10:50:00
12819,2026,Spring,2026-sp,YDSH,102,Beginning Yiddish II,Continuation of YDSH 101 focusing on comprehen...,4 hours.,NaN,NaN,...,ONL,11:00 AM,12:00 PM,F,NaN,NaN,"Blazek, K",4.0,11:00:00,12:00:00
12820,2026,Spring,2026-sp,YDSH,102,Beginning Yiddish II,Continuation of YDSH 101 focusing on comprehen...,4 hours.,NaN,NaN,...,ONL,12:00 PM,01:30 PM,TR,NaN,NaN,"Blazek, K",4.0,12:00:00,13:30:00
12821,2026,Spring,2026-sp,YDSH,104,Intermediate Yiddish II,Continuation of YDSH 103. Prerequisite: YDSH 1...,4 hours.,Prerequisite: YDSH 103 or equivalent placement...,NaN,...,ONL,09:00 AM,10:00 AM,R,NaN,NaN,"Blazek, K",4.0,09:00:00,10:00:00
